# Error-Driven Evaluation: Contextual Biasing Where Whisper Actually Fails

**Why this notebook exists.** The previous evaluation (`whisper_trie_sonar_biasing_pipeline.ipynb`)
came back *inconclusive by ceiling*: on LibriSpeech `validation.clean`, baseline recall on the mined
"rare" words was **0.983** (57/58 already correct), all three conditions produced identical numbers,
and 0 transcripts differed between trie-only and trie+SONAR. The instrument worked; the test bench had
nothing on it. Text-rarity is a bad proxy for acoustic difficulty on clean read speech.

**The fix implemented here — three structural changes:**

1. **Harder audio:** LibriSpeech `test.other` — noisier recordings, harder speakers; `whisper-base`
   errors 2–3× more often, so there is something to recover.
2. **Error-driven target mining (the core idea):** Phase 1 transcribes a large pool of utterances with
   *plain* Whisper **before any biasing list exists**. Targets are mined from reference words the
   baseline demonstrably **deleted or substituted**. Baseline recall on targets is therefore ≈ 0 *by
   construction* — maximal headroom, standard practice in the deep-biasing literature.
3. **Gate discriminability diagnostic:** before any biased decoding, a SONAR-only check answers
   *"could* the semantic gate work in principle?" — does cos(word embedding, utterance embedding)
   separate in-context targets from distractors? If this oracle check fails, condition C cannot
   succeed for any (δ, λ), and we learn *why* instead of just *that*.

## The three conditions (unchanged)

| | Decoding | Question it answers |
|---|---|---|
| **A** | plain beam search (cached from Phase 1) | how bad is it unaided? |
| **B** | trie boost δ per continuing token | does classic shallow fusion recover the errors? |
| **C** | boost = δ + λ·max(0, cos(state@W, word SONAR emb)) | does semantic gating beat B on the recall / false-alarm trade-off? |

The biasing list again mixes **targets** (mined errors) with **distractors** (rare words guaranteed
absent from dev/eval audio) — the false-alarm control that gives C something to prove.

## Honest caveats, stated up front

- **Selection bias is the point, but cuts both ways:** targets are baseline errors, so *any* recovery
  is a win — but some errors are unrecoverable (mumbled, cut off), so recall 1.0 is not the bar.
  B-vs-C and false alarms are the comparisons that matter, not absolute recall.
- **Statistics:** differences need occurrences to mean anything; the notebook prints
  `target_occurrences` — treat gaps as noise unless they involve ≥ ~20 occurrences.
- Tuning (δ, λ) happens on dev utterances only; eval is untouched until the final table.

**Runtime:** Phase 1 dominates (~250 baseline transcriptions). On CUDA expect ~15–30 min total;
`FAST` mode (auto on CPU) shrinks everything to a demo-scale run.


---
## Section 0.1 — Dependencies (identical stack to notebook 2)


In [ ]:
# Run once, then restart the kernel.
# %pip install -U torch transformers datasets scikit-learn scipy pandas matplotlib jiwer soundfile librosa
# %pip install sonar-space


## Section 0.2 — Imports, seed, device


In [ ]:
import os, math, random, re, json, itertools
from collections import Counter
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import jiwer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)


## Section 0.3 — Configuration

New knobs versus notebook 2:
- `n_pool` — Phase 1 baseline pool size. Bigger pool → more mined errors → more targets. 250 gives a
  comfortable target list on `test.other`.
- `n_dev_utts` / `n_eval_utts` — chosen **from utterances that contain a mined target**, so every
  utterance in the experiment has headroom (that was the fatal flaw last time).
- `max_audio_s` — utterances longer than Whisper's 30 s window are skipped (a handful in test.other).
- `delta_grid` extended upward (2–4): with real headroom, stronger boosts are worth probing; the
  false-alarm metric now polices the upper end instead of nothing-to-gain apathy.
- `min_word_len` dropped to 4 — error mining already guarantees difficulty, so we can admit shorter
  words that rarity-mining could not risk.


In [ ]:
FAST = (DEVICE == "cpu")

CONFIG = {
    "whisper_model": "openai/whisper-base",
    "best_layer": 4,
    "w_path": "whisper_to_sonar_W.pt",
    "n_pool":     60 if FAST else 250,
    "n_dev_utts":  8 if FAST else 15,
    "n_eval_utts": 20 if FAST else 60,
    "n_targets_max": 30 if FAST else 60,
    "n_distractors": 30 if FAST else 60,
    "max_audio_s": 29.0,
    "beams": 3 if FAST else 5,
    "delta_grid":  [2.0, 4.0] if FAST else [1.0, 2.0, 3.0, 4.0],
    "lambda_grid": [4.0] if FAST else [2.0, 4.0, 8.0],
    "min_word_len": 4,
    "max_doc_freq": 3,
}
print("FAST mode:", FAST)
CONFIG


---
## Section 1 — Frozen Whisper + alignment matrix `W`

Identical to notebook 2 (see there for the full rationale): load `whisper-base`, pin the English
transcription prompt, load `whisper_to_sonar_W.pt` — or refit a compact substitute if it is missing,
so this notebook stands alone. The refit is the same silence-conditioned layer-4 recipe validated in
notebook 1.


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import LogitsProcessor, LogitsProcessorList

processor = WhisperProcessor.from_pretrained(CONFIG["whisper_model"])
whisper = (WhisperForConditionalGeneration
           .from_pretrained(CONFIG["whisper_model"]).to(DEVICE).eval())
tok = processor.tokenizer
tok.set_prefix_tokens(language="english", task="transcribe")
N_PREFIX = len(tok.prefix_tokens)
print(f"Whisper ready: d_model={whisper.config.d_model}")

W = None
if os.path.exists(CONFIG["w_path"]):
    W = torch.load(CONFIG["w_path"], map_location="cpu").float()
    print(f"Loaded W {tuple(W.shape)} from {CONFIG['w_path']}")
else:
    print("!! No W found — run the refit cell below.")


In [ ]:
if W is None:
    from datasets import load_dataset
    from sklearn.linear_model import Ridge
    from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    sents = []
    for line in ds["text"]:
        line = line.strip().replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")
        if not line or line.startswith("="):
            continue
        for s in re.split(r"(?<=[.!?]) +", line):
            if 40 <= len(s.strip()) <= 200:
                sents.append(s.strip())
    sents = list(dict.fromkeys(sents))
    random.Random(SEED).shuffle(sents)
    sents = sents[:800]

    sil = processor.feature_extractor(np.zeros(16000 * 30, dtype=np.float32),
                                      sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    with torch.no_grad():
        enc0 = whisper.get_encoder()(sil).last_hidden_state
        X = []
        for i in range(0, len(sents), 16):
            e = tok(sents[i:i+16], return_tensors="pt", padding=True)
            ids, attn = e.input_ids.to(DEVICE), e.attention_mask.to(DEVICE)
            out = whisper(encoder_outputs=(enc0.expand(ids.shape[0], -1, -1),),
                          decoder_input_ids=ids, output_hidden_states=True, return_dict=True)
            hs = out.decoder_hidden_states[CONFIG["best_layer"]]
            m = attn.clone(); m[:, :N_PREFIX] = 0
            m.scatter_(1, attn.sum(1, keepdim=True) - 1, 0)
            m = m.unsqueeze(-1).float()
            X.append(F.normalize((hs * m).sum(1) / m.sum(1).clamp(min=1), dim=-1).cpu())
    X = torch.cat(X).numpy()
    t2v_fit = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder",
                                           tokenizer="text_sonar_basic_encoder",
                                           device=torch.device("cpu"))
    Y = F.normalize(t2v_fit.predict(sents, source_lang="eng_Latn", batch_size=64).float(),
                    dim=-1).numpy()
    W = torch.from_numpy(Ridge(alpha=10.0, fit_intercept=False).fit(X, Y).coef_.T).float()
    torch.save(W, CONFIG["w_path"])
    print(f"Refit W {tuple(W.shape)} → saved.")
else:
    print("W loaded — refit skipped.")


---
## Section 2 — SONAR pipeline (embeds biasing words *and* utterance texts for the diagnostic)


In [ ]:
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

t2vec = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder",
                                     tokenizer="text_sonar_basic_encoder",
                                     device=torch.device("cpu"))

def sonar_embed(texts):
    with torch.no_grad():
        e = t2vec.predict(list(texts), source_lang="eng_Latn", batch_size=64).float()
    return F.normalize(e, dim=-1)

print("SONAR ready.")


---
## Section 3 — Streaming the `test.other` pool

Line-by-line:
1. `test.other` is LibriSpeech's *hard* split (noisier recordings, less canonical speakers) — chosen
   precisely because `whisper-base` makes real mistakes on it. The dummy-dataset fallback keeps the
   notebook runnable offline, with a printed warning that it is demo-only.
2. Audio is materialized **once** into `pool` (a list of `{audio, text}` dicts) — Phase 1, the sweep,
   and the final eval all reuse these arrays instead of re-decoding the stream.
3. Utterances longer than `max_audio_s` are skipped (they would truncate inside Whisper's 30 s window
   and pollute WER with truncation errors that no biasing can fix).
4. `norm()` — the single normalization shared by mining and metrics, as in notebook 2.


In [ ]:
from datasets import load_dataset, Audio

def norm(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

try:
    stream = load_dataset("openslr/librispeech_asr", "other", split="test", streaming=True)
    DATA_NAME = "LibriSpeech test.other (streaming)"
except Exception as e:
    print("test.other unavailable → dummy fallback:", repr(e)[:120])
    stream = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean",
                          split="validation", streaming=True)
    DATA_NAME = "librispeech_asr_dummy (demo only — results not meaningful)"
stream = stream.cast_column("audio", Audio(sampling_rate=16000))

def get_audio(sample):
    a = sample["audio"]
    if isinstance(a, dict) and a.get("array") is not None:
        return np.asarray(a["array"], dtype=np.float32)
    if hasattr(a, "get_all_samples"):
        return a.get_all_samples().data.numpy().astype(np.float32).flatten()
    import soundfile as sf
    return sf.read(a["path"], dtype="float32")[0]

pool, skipped = [], 0
for s in stream:
    audio = get_audio(s)
    if len(audio) > 16000 * CONFIG["max_audio_s"]:
        skipped += 1
        continue
    pool.append({"audio": audio, "text": s["text"]})
    if len(pool) >= CONFIG["n_pool"]:
        break

print(f"{DATA_NAME}\npool={len(pool)} utterances ({skipped} skipped for length)")
print("sample ref:", pool[0]["text"][:90])


---
## Section 4 — Phase 1: the baseline pass (the mining substrate)

The plain-Whisper transcription of every pool utterance. Three later consumers, one pass:
1. **Target mining** (next section) reads its *errors*;
2. **Condition A** numbers are simply looked up here — never recomputed;
3. The **dev sweep**'s baseline row likewise.

`transcribe()` is notebook 2's harness verbatim (feature-extract → `generate` with pinned
language/task and beams → decode); biasing arguments come later. This cell is the longest-running in
the notebook — a progress line every 10 utterances.


In [ ]:
@torch.no_grad()
def transcribe(audio, delta=0.0, lam=0.0, bias=None):
    feats = processor.feature_extractor(
        audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    kwargs = dict(language="en", task="transcribe",
                  num_beams=CONFIG["beams"], use_cache=(lam == 0.0))
    proc = None
    if bias is not None and (delta > 0 or lam > 0):
        capture.hidden = None
        proc = TrieSonarBiasProcessor(*bias, delta=delta, lam=lam, n_prefix=N_PREFIX)
        kwargs["logits_processor"] = LogitsProcessorList([proc])
    ids = whisper.generate(feats, **kwargs)
    return tok.decode(ids[0], skip_special_tokens=True).strip(), proc

print("Phase 1: baseline transcription of the pool ...")
for i, u in enumerate(pool):
    u["base"], _ = transcribe(u["audio"])
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(pool)}")

base_wer_pool = jiwer.wer([norm(u['text']) for u in pool],
                          [norm(u['base']) for u in pool])
print(f"pool baseline WER = {base_wer_pool:.3f}   (clean-split ceiling problem solved if ≫ 0.03)")


---
## Section 5 — Error-driven target mining

Line-by-line:
1. For every pool utterance, `jiwer.process_words` aligns reference vs. baseline hypothesis; every
   reference word inside a `substitute` or `delete` chunk is a **miss** — collected with counts in
   `err_counter`, alongside document frequency `df` over the pool.
2. **Target filter:** missed at least once, ≥ 4 chars, alphabetic, document frequency ≤ 3. The last
   condition keeps the list in "contextual biasing" territory — rare entities and content words, not
   function words the model fumbled once ("their/there" noise).
3. The printed table shows each candidate with `missed` vs `total` reference occurrences in the pool —
   words missed *every* time they occur are the hardest cases; the mix is visible up front.

The contrast with notebook 2's mining is the whole story: there, targets were *guessed* from text
statistics (and Whisper knew them all); here every target is a *certified* baseline failure.


In [ ]:
err_counter, occ_counter = Counter(), Counter()
df = Counter()
for u in pool:
    rw = norm(u["text"]).split()
    df.update(set(rw))
    occ_counter.update(rw)
    out = jiwer.process_words([norm(u["text"])], [norm(u["base"])])
    for ch in out.alignments[0]:
        if ch.type in ("substitute", "delete"):
            for i in range(ch.ref_start_idx, ch.ref_end_idx):
                err_counter[rw[i]] += 1

cand = {w: c for w, c in err_counter.items()
        if len(w) >= CONFIG["min_word_len"] and w.isalpha()
        and df[w] <= CONFIG["max_doc_freq"]}

mined = (pd.DataFrame([{"word": w, "missed": c, "total_occ": occ_counter[w]}
                       for w, c in cand.items()])
         .sort_values(["missed", "total_occ"], ascending=[False, True])
         .reset_index(drop=True))
print(f"{len(mined)} error-mined target candidates")
mined.head(15)


---
## Section 6 — Utterance selection, dev/eval split, final biasing list

Line-by-line:
1. Keep only utterances whose reference contains ≥ 1 mined candidate — every experimental utterance
   now has certified headroom (the structural fix for last run's 0.983-recall ceiling).
2. Shuffle (pinned seed), first `n_dev_utts` → dev, next `n_eval_utts` → eval.
3. **Targets** = mined candidates that occur in dev∪eval references, capped at `n_targets_max`
   (prioritized by miss count — the hardest words first).
4. **Distractors** = words passing the *same* rarity/shape filter, drawn from pool utterances **outside**
   dev∪eval and required absent from every dev/eval reference — same adversarial role as before.
5. `E_bias` — SONAR embeddings for the whole list, index-aligned with `bias_words`;
   `TARGETSET`/`HOTSET` drive the metrics exactly as in notebook 2.


In [ ]:
cand_set = set(cand)
sel = [u for u in pool if set(norm(u["text"]).split()) & cand_set]
random.Random(SEED).shuffle(sel)
dev_set  = sel[:CONFIG["n_dev_utts"]]
eval_set = sel[CONFIG["n_dev_utts"]:CONFIG["n_dev_utts"] + CONFIG["n_eval_utts"]]
print(f"utterances with ≥1 mined error: {len(sel)} → dev={len(dev_set)} eval={len(eval_set)}")

de_vocab = set(w for u in dev_set + eval_set for w in norm(u["text"]).split())
targets = [w for w in mined["word"] if w in de_vocab][:CONFIG["n_targets_max"]]

used = {id(u) for u in dev_set + eval_set}
outside = [u for u in pool if id(u) not in used]
distr_cand = [w for u in outside for w in norm(u["text"]).split()
              if len(w) >= CONFIG["min_word_len"] and w.isalpha()
              and df[w] <= CONFIG["max_doc_freq"] and w not in de_vocab]
distractors = list(dict.fromkeys(distr_cand))[:CONFIG["n_distractors"]]

bias_words = targets + distractors
TARGETSET, HOTSET = set(targets), set(bias_words)
print(f"targets({len(targets)}):", targets[:8], "...")
print(f"distractors({len(distractors)}):", distractors[:8], "...")

E_bias = sonar_embed(bias_words).to(DEVICE)
print("bias embeddings:", tuple(E_bias.shape))


---
## Section 6b — Gate discriminability diagnostic (new, and worth the two minutes)

Before spending an hour on biased decoding, ask the cheap oracle question: **if the projection from
Whisper states were perfect, would SONAR similarity even separate targets from distractors?**

Line-by-line:
1. Embed every *eval reference sentence* with SONAR (`U`) — this is the *oracle* utterance context;
   the runtime system only has the noisier `state @ W` estimate, so whatever separation exists here is
   an **upper bound** on what the gate can exploit.
2. `in_ctx` — cos(target embedding, embedding of the utterance that *contains* it): what the gate sees
   when it should say yes.
3. `out_ctx` — cos(target embedding, other utterances): same word, wrong context.
4. `distr` — cos(distractor embedding, every eval utterance): what the gate must say no to.
5. The histogram is the verdict: **clear separation of `in_ctx` from the other two** → the semantic
   signal exists and condition C has a fighting chance. **Overlapping blobs** → word-level SONAR
   embeddings don't carry utterance-level context; expect C ≈ B *regardless of λ*, and the upgrade
   path is phrase/topic-level biasing entries, not hyperparameters. Either way, you interpret the
   final table knowing which world you are in.


In [ ]:
U = sonar_embed([norm(u["text"]) for u in eval_set])          # [n_eval, 1024] oracle contexts
E_t = sonar_embed(targets) if targets else torch.zeros(0, 1024)
E_d = sonar_embed(distractors) if distractors else torch.zeros(0, 1024)

in_ctx, out_ctx = [], []
for j, u in enumerate(eval_set):
    words = set(norm(u["text"]).split())
    for i, w in enumerate(targets):
        c = float(E_t[i] @ U[j])
        (in_ctx if w in words else out_ctx).append(c)
distr = (E_d @ U.T).flatten().tolist()

plt.figure(figsize=(8, 4))
plt.hist(out_ctx, bins=40, alpha=0.5, density=True, label=f"target, wrong utt (μ={np.mean(out_ctx):.3f})")
plt.hist(distr,   bins=40, alpha=0.5, density=True, label=f"distractor (μ={np.mean(distr):.3f})")
plt.hist(in_ctx,  bins=40, alpha=0.7, density=True, label=f"target, own utt (μ={np.mean(in_ctx):.3f})")
plt.xlabel("cos(word emb, utterance emb)"); plt.ylabel("density")
plt.title("Oracle gate discriminability (SONAR only — no Whisper involved)")
plt.legend(); plt.tight_layout(); plt.show()

sep = np.mean(in_ctx) - np.mean(distr)
print(f"separation (in-context − distractor) = {sep:.3f}"
      f"  → {'gate has signal' if sep > 0.05 else 'WEAK — expect C ≈ B; see Section 12'}")


---
## Section 7 — Biasing trie (verbatim from notebook 2)

One BPE token per edge; each word inserted under 4 surface forms (±capitalization × ±leading space,
Whisper's context-sensitive BPE demands it) sharing one hotword id; `hids` at every node link edges
back to SONAR embeddings. Full rationale in notebook 2 §5. The unit test re-asserts the casing
behavior against *this* run's vocabulary.


In [ ]:
class TrieNode:
    __slots__ = ("children", "hids")
    def __init__(self):
        self.children = {}
        self.hids = set()

class BiasTrie:
    def __init__(self):
        self.root = TrieNode()
        self.max_depth = 0
    def insert(self, token_ids, hid):
        node = self.root
        for t in token_ids:
            node = node.children.setdefault(t, TrieNode())
            node.hids.add(hid)
        self.max_depth = max(self.max_depth, len(token_ids))
    def walk(self, seq):
        node = self.root
        for t in seq:
            node = node.children.get(t)
            if node is None:
                return None
        return node

trie = BiasTrie()
for hid, w in enumerate(bias_words):
    for form in {w, w.capitalize()}:
        for surface in (form, " " + form):
            ids = tok.encode(surface, add_special_tokens=False)
            if ids:
                trie.insert(ids, hid)
print(f"Trie: {len(bias_words)} words, max_depth={trie.max_depth}")
assert trie.walk(tok.encode(" " + bias_words[0], add_special_tokens=False)) is not None
assert trie.walk(tok.encode(" zzqx", add_special_tokens=False)) is None
print("unit test passed")


---
## Section 8 — State-capture hook + `TrieSonarBiasProcessor` (verbatim from notebook 2)

Hook on `decoder.layers[best_layer − 1]` (module index 3 = `hidden_states[4]`); condition C decodes
with `use_cache=False` so the captured tensor always spans the full hypothesis — pooled, projected
through `W`, cosine'd against `E_bias`, and folded into per-token boosts `δ + λ·max(0, cos)` via the
trie suffix walk. Every design decision is documented in notebook 2 §6–7; this cell is a faithful copy
so the two notebooks stay independently runnable.


In [ ]:
class StateCapture:
    def __init__(self):
        self.hidden = None
    def __call__(self, module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        self.hidden = h.detach()

capture = StateCapture()
hook_handle = whisper.model.decoder.layers[CONFIG["best_layer"] - 1].register_forward_hook(capture)
W_DEV = W.to(DEVICE)

class TrieSonarBiasProcessor(LogitsProcessor):
    def __init__(self, trie, E_bias, W, capture, delta, lam, n_prefix):
        self.trie, self.E = trie, E_bias
        self.W = W.to(E_bias.device)
        self.capture = capture
        self.delta, self.lam = float(delta), float(lam)
        self.n_prefix = n_prefix
        self.stats = {"steps": 0, "boosted_steps": 0}

    def __call__(self, input_ids, scores):
        self.stats["steps"] += 1
        beams = input_ids.shape[0]
        sims = None
        if self.lam > 0 and self.capture.hidden is not None:
            hs = self.capture.hidden
            if hs.shape[0] == beams and hs.shape[1] > self.n_prefix:
                pooled = F.normalize(hs[:, self.n_prefix:, :].mean(dim=1), dim=-1)
                proj = F.normalize(pooled.float() @ self.W, dim=-1)
                sims = (proj @ self.E.T).cpu()
        for b in range(beams):
            gen = input_ids[b, self.n_prefix:].tolist()
            bonus = {}
            lo = max(0, len(gen) - self.trie.max_depth + 1)
            for start in range(lo, len(gen) + 1):
                node = self.trie.walk(gen[start:])
                if node is None:
                    continue
                for t, child in node.children.items():
                    val = self.delta
                    if sims is not None and child.hids:
                        s = max(sims[b, h].item() for h in child.hids)
                        val += self.lam * max(0.0, s)
                    if val > bonus.get(t, 0.0):
                        bonus[t] = val
            if bonus:
                self.stats["boosted_steps"] += 1
                idx = torch.tensor(list(bonus.keys()), device=scores.device)
                val = torch.tensor(list(bonus.values()), device=scores.device, dtype=scores.dtype)
                scores[b].index_add_(0, idx, val)
        return scores

BIAS = (trie, E_bias, W_DEV, capture)
print("Processor ready.")


---
## Section 9 — Smoke test on a certified failure

Unlike last time, we can pick an utterance where the baseline **provably missed a target** — the diff
between A and B/C should now be visible to the naked eye. Also prints the boost counters (machinery
engaged) and the missed words being hunted.


In [ ]:
def missed_targets(u):
    rw, hw = set(norm(u["text"]).split()), set(norm(u["base"]).split())
    return sorted((rw & TARGETSET) - hw)

smoke = next((u for u in eval_set if missed_targets(u)), eval_set[0])
print("REF :", smoke["text"])
print("miss:", missed_targets(smoke), "\n")
print(f"{'A baseline':26s}: {smoke['base']}")
for label, d, l in [("B trie δ=3", 3.0, 0.0), ("C trie+SONAR δ=3 λ=4", 3.0, 4.0)]:
    hyp, p = transcribe(smoke["audio"], d, l, BIAS)
    print(f"{label:26s}: {hyp}   [boosted {p.stats['boosted_steps']}/{p.stats['steps']} steps]")


---
## Section 10 — Metrics (notebook 2's `score_corpus`, unchanged)

B-WER/U-WER by reference-word attribution, recall over target occurrences from `equal` chunks, false
alarms from inserted/substituted-in biasing words. Full attribution rules in notebook 2 §9. The only
addition is `run_biased()`, which iterates cached-audio utterance dicts.


In [ ]:
def score_corpus(refs, hyps):
    R, H = [norm(r) for r in refs], [norm(h) for h in hyps]
    out = jiwer.process_words(R, H)
    hits = occ = b_err = u_err = b_ref = u_ref = fa = 0
    for rtxt, htxt, chunks in zip(R, H, out.alignments):
        rw, hw = rtxt.split(), htxt.split()
        for w in rw:
            if w in HOTSET: b_ref += 1
            else:           u_ref += 1
            if w in TARGETSET: occ += 1
        for ch in chunks:
            if ch.type == "equal":
                hits += sum(1 for i in range(ch.ref_start_idx, ch.ref_end_idx)
                            if rw[i] in TARGETSET)
            elif ch.type in ("substitute", "delete"):
                for i in range(ch.ref_start_idx, ch.ref_end_idx):
                    if rw[i] in HOTSET: b_err += 1
                    else:               u_err += 1
                if ch.type == "substitute":
                    fa += sum(1 for j in range(ch.hyp_start_idx, ch.hyp_end_idx)
                              if hw[j] in HOTSET and hw[j] not in rw)
            elif ch.type == "insert":
                for j in range(ch.hyp_start_idx, ch.hyp_end_idx):
                    if hw[j] in HOTSET:
                        b_err += 1; fa += 1
                    else:
                        u_err += 1
    return {"wer": out.wer,
            "b_wer": b_err / b_ref if b_ref else float("nan"),
            "u_wer": u_err / u_ref if u_ref else float("nan"),
            "recall": hits / occ if occ else float("nan"),
            "false_alarms": fa, "target_occurrences": occ}

def run_biased(samples, delta, lam, tag=""):
    hyps = []
    for i, u in enumerate(samples):
        hyp, _ = transcribe(u["audio"], delta, lam, BIAS)
        hyps.append(hyp)
        if (i + 1) % 10 == 0:
            print(f"  {tag} {i + 1}/{len(samples)}")
    return hyps

print("Metrics ready.")


---
## Section 11 — Dev sweep (baseline row is free — cached from Phase 1)

Same two-stage protocol as notebook 2: δ swept with λ=0, `δ*` = best recall subject to WER ≤ 1.05×
baseline; then λ swept at `δ*`. With genuine headroom the sweep should finally show its expected
shape — recall climbing with δ until WER/false alarms object.


In [ ]:
dev_refs = [u["text"] for u in dev_set]
rows = [{"cond": "A", "delta": 0.0, "lam": 0.0,
         **score_corpus(dev_refs, [u["base"] for u in dev_set])}]
base_wer = rows[0]["wer"]
print(f"dev baseline (cached): wer={base_wer:.3f} recall={rows[0]['recall']:.3f}")

for d in CONFIG["delta_grid"]:
    print(f"dev B δ={d} ...")
    rows.append({"cond": "B", "delta": d, "lam": 0.0,
                 **score_corpus(dev_refs, run_biased(dev_set, d, 0.0, f"B δ={d}"))})

dev_df = pd.DataFrame(rows)
c = dev_df[(dev_df["cond"] == "B") & (dev_df["wer"] <= base_wer * 1.05)]
DELTA = float((c if len(c) else dev_df[dev_df["cond"] == "B"])
              .sort_values(["recall", "wer"], ascending=[False, True]).iloc[0]["delta"])
print(f"δ* = {DELTA}")

for l in CONFIG["lambda_grid"]:
    print(f"dev C δ={DELTA} λ={l} ...")
    rows.append({"cond": "C", "delta": DELTA, "lam": l,
                 **score_corpus(dev_refs, run_biased(dev_set, DELTA, l, f"C λ={l}"))})

dev_df = pd.DataFrame(rows)
c = dev_df[(dev_df["cond"] == "C") & (dev_df["wer"] <= base_wer * 1.05)]
LAM = float((c if len(c) else dev_df[dev_df["cond"] == "C"])
            .sort_values(["recall", "wer"], ascending=[False, True]).iloc[0]["lam"])
print(f"λ* = {LAM}")
dev_df.round(3)


---
## Section 12 — Final evaluation on the untouched eval set

Condition A comes straight from the Phase 1 cache; B and C decode with the tuned (δ\*, λ\*). Then the
per-target recovery table: for every target word, how many of its eval-reference occurrences each
condition transcribed correctly — the word-level view that shows *which* errors biasing fixed, not
just how many.


In [ ]:
eval_refs = [u["text"] for u in eval_set]
transcripts = {"A baseline": [u["base"] for u in eval_set]}
final_rows = [{"condition": "A baseline",
               **score_corpus(eval_refs, transcripts["A baseline"])}]

for label, d, l in [(f"B trie δ={DELTA}", DELTA, 0.0),
                    (f"C trie+SONAR δ={DELTA} λ={LAM}", DELTA, LAM)]:
    print(f"eval: {label} ...")
    transcripts[label] = run_biased(eval_set, d, l, label)
    final_rows.append({"condition": label, **score_corpus(eval_refs, transcripts[label])})

final_df = pd.DataFrame(final_rows).set_index("condition")
final_df.round(3)


In [ ]:
def recovered(word, hyps):
    n = 0
    for u, h in zip(eval_set, hyps):
        r_c = norm(u["text"]).split().count(word)
        if r_c:
            n += min(r_c, norm(h).split().count(word))
    return n

per_word = []
for w in targets:
    occ = sum(norm(u["text"]).split().count(w) for u in eval_set)
    if occ == 0:
        continue
    per_word.append({"target": w, "occ": occ,
                     "A": recovered(w, transcripts["A baseline"]),
                     "B": recovered(w, transcripts[final_df.index[1]]),
                     "C": recovered(w, transcripts[final_df.index[2]])})
per_word_df = pd.DataFrame(per_word).sort_values("occ", ascending=False)
print("per-target recoveries (occurrences vs correctly transcribed):")
per_word_df


### Visual summary + B-vs-C transcript diffs

Same two panels as notebook 2 — but now read against real headroom: baseline recall should sit *low*,
and the interesting gaps are (B−A) for the trie and (C−B) for the semantic gate, with the red
false-alarm line as the cost axis.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
final_df[["wer", "b_wer", "u_wer"]].plot.bar(ax=axes[0], rot=15)
axes[0].set_ylabel("error rate"); axes[0].set_title("WER decomposition (eval)")

ax2 = axes[1]
final_df["recall"].plot.bar(ax=ax2, color="tab:green", rot=15)
ax2.set_ylabel("target recall", color="tab:green")
ax3 = ax2.twinx()
ax3.plot(range(len(final_df)), final_df["false_alarms"], "ro-")
ax3.set_ylabel("false alarms (count)", color="tab:red")
ax2.set_title("Recall vs false alarms (eval)")
plt.tight_layout(); plt.show()

names = list(final_df.index)
diff = [(u["text"], a, b) for u, a, b in
        zip(eval_set, transcripts[names[1]], transcripts[names[2]])
        if norm(a) != norm(b)]
print(f"{len(diff)} utterances differ between B and C:")
for r, a, b in diff[:4]:
    print("\nREF:", r, "\n B :", a, "\n C :", b)


---
## Section 13 — Save artifacts


In [ ]:
dev_df.to_csv("errdriven_dev_sweep.csv", index=False)
final_df.to_csv("errdriven_final_results.csv")
per_word_df.to_csv("errdriven_per_target.csv", index=False)
pd.DataFrame({"ref": eval_refs, **transcripts}).to_csv("errdriven_eval_transcripts.csv", index=False)
with open("errdriven_meta.json", "w") as f:
    json.dump({"config": CONFIG, "dataset": DATA_NAME,
               "delta_star": DELTA, "lambda_star": LAM,
               "pool_baseline_wer": base_wer_pool,
               "gate_separation": float(np.mean(in_ctx) - np.mean(distr)),
               "n_targets": len(targets), "n_distractors": len(distractors)}, f, indent=2)
hook_handle.remove()
print("Saved: errdriven_{dev_sweep,final_results,per_target,eval_transcripts}.csv + errdriven_meta.json")


---
## Section 14 — Reading the results (updated rubric)

**First, check the two preconditions this notebook engineered:**
- Baseline eval recall should now be **low** (≪ 0.9). If it is still high, mining leaked easy words —
  inspect `mined` for normalization artifacts.
- The Section 6b diagnostic told you whether the semantic gate *could* work. Interpret C through that
  lens: with weak oracle separation, C ≈ B is the *predicted* outcome and indicts word-level
  embeddings, not the architecture.

**Then the verdicts:**
1. **B ≫ A on recall, C > B and/or fewer false alarms, U-WER flat** — full thesis confirmed on hard
   audio. Next: scale eval (all 2,939 test.other utts), phrase-level entries, `whisper-small` + its
   refit `W`, then the production port (incremental pooled states, KV cache on).
2. **B ≫ A but C ≈ B, diagnostic showed signal** — the trie works; the runtime projection is the weak
   link (silence-fitted `W`, noisy prefix states). Upgrades: refit `W` on *real-audio* states
   (transcribe LibriSpeech, pool states from those runs — the harness here already produces them), or
   λ-normalize similarities (softmax over the list) so small cosine gaps still gate.
3. **B ≫ A but C ≈ B, diagnostic weak** — word-level SONAR embeddings are the bottleneck. Switch
   biasing entries to phrases/topic sentences ("a lecture about {word} and {word}...") or per-utterance
   context prompts; the trie and `W` machinery carry over unchanged.
4. **B ≈ A even here** — with certified-failure targets this now *is* informative: δ-level shallow
   fusion cannot rescue these words (likely acoustically absent). Inspect `per_word_df` — words with
   `occ` high and `B = 0` are candidates for listening; if the audio truly contains them, the boost is
   being out-scored by the acoustic path and deeper integration (TCPGen-style pointer networks, i.e.
   *trained* biasing) is the honest conclusion.

**Statistical footnote:** with ~50–150 target occurrences, differences of a few hits are noise. The
per-word table plus `target_occurrences` keep the denominators visible; for a publishable claim, run
the full split and bootstrap the recall difference.
